In [ ]:
import glob
import os
import sys

import einops
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from kornia.geometry.transform import resize
from matplotlib import pyplot as plt
from PIL import Image
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

sys.path.append("your_path")
from src.tasks.pretrain.backbone import (
    get_dinov2,
    get_eva02,
    get_flair,
    get_keepfit,
    get_retclip,
    get_retfound,
    get_urfound,
)
from src.tasks.pretrain.encoder import SSLEncoder
from src.tasks.pretrain.utils import (
    compute_pca_features,
    convert_attn_to_heatmap,
    min_max_scale_2d,
)

# Set random seeds for reproducibility
torch.manual_seed(123)
np.random.seed(123)

# Set CUDA device
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Change this to the GPU you want to use

# Model Loader

In [ ]:
def get_freqmae(weights_path):
    encoder = SSLEncoder(
        model_name="timm/vit_base_patch14_reg4_dinov2.lvd142m",
        algorithm="freqmae",
        pretrained=True,
        ckpt_path=weights_path,
        mask_ratio=0.0,
        img_size=224,
        patch_size=14,
        reg_tokens=4,
        in_chans=3,
        embed_dim=768,
        depth=12,
        num_heads=12,
        mlp_ratio=4.0,
        drop_rate=0.0,
        pos_drop_rate=0.0,
        patch_drop_rate=0.0,
        proj_drop_rate=0.0,
        attn_drop_rate=0.0,
        drop_path_rate=0.0,
        standardize=True,
        dynamic_img_size=True,
        norm_layer=nn.LayerNorm,
    )
    return encoder.backbone

# Data Loader

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, img_dir, transform=None, img_size=224):
        """
        Args:
            img_dir (str): Directory with all the images.
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        self.img_dir = img_dir
        self.img_paths = sorted(glob.glob(os.path.join(img_dir, "*.*")))
        self.transform = transform or transforms.Compose(
            [
                transforms.ToTensor(),
                transforms.Resize((img_size, img_size), antialias=True),
            ]
        )

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image


# 이미지 디렉토리 경로 설정
img_dir = "your_path/src/tasks/pretrain/visualization/imgs"

# 데이터셋 및 데이터로더 생성
dataset = ImageDataset(img_dir, img_size=224)
dataloader = DataLoader(dataset, batch_size=8, shuffle=False)

# 모든 이미지를 하나의 텐서로 로드
all_images = []
for batch in dataloader:
    all_images.append(batch)

# 모든 배치를 하나의 텐서로 결합 (B, C, H, W)
data = torch.cat(all_images, dim=0)
print(f"Loaded images tensor shape: {data.shape}")

# 시각화를 위해 첫 번째 이미지 확인 (선택 사항)
if len(data) > 0:
    print(f"Number of images loaded: {len(data)}")
    print(f"Image dimensions: {data[0].shape}")

# Main Functions

In [ ]:
def patch_pca_feats(patch_feats, patch_size, img, mode="nearest"):
    if patch_feats.ndim == 3:
        B, N, D = patch_feats.shape
        patch_feats = einops.rearrange(patch_feats, "b (h w) c -> (b h w) c", h=int(np.sqrt(N)))

    elif patch_feats.ndim == 4:
        B, D, H, W = patch_feats.shape
        N = H * W
        patch_feats = einops.rearrange(patch_feats, "b c h w -> (b h w) c")

    fg_mask = img.mean(dim=1)
    th = 0.0868832016680601 * fg_mask.amax(dim=(1, 2), keepdim=True)
    fg_mask = (fg_mask > th).float()
    fg_mask = F.interpolate(
        fg_mask.unsqueeze(1), size=(int(np.sqrt(N)), int(np.sqrt(N))), mode="nearest"
    )
    fg_mask = fg_mask.squeeze(1)
    # fg_mask = einops.rearrange(fg_mask, "b c h w -> (b h w) c", c=1).bool().repeat(1, D)

    # PCA 특징 계산
    pca_feats = compute_pca_features(patch_feats, fg_mask=fg_mask)  # ((b h w) c)
    h = w = int(np.sqrt(N))
    pca_feats = einops.rearrange(pca_feats, "(b h w) c -> b h w c", h=h, w=w).permute(0, 3, 1, 2)

    # 업샘플링
    target_size = (h * patch_size, w * patch_size)
    pca_feats = torch.nn.functional.interpolate(
        pca_feats, size=target_size, mode=mode
    )  # (B, 3, H, W)

    return pca_feats.clamp(0, 1)


def patch_norm_feats(patch_feats, patch_size, mode="bilinear"):
    if patch_feats.ndim == 3:
        B, N, D = patch_feats.shape

    elif patch_feats.ndim == 4:
        B, C, H, W = patch_feats.shape
        N = H * W
        patch_feats = einops.rearrange(patch_feats, "b c h w -> b (h w) c")

    # 패치 특징의 L2 norm 계산 (B, N)
    feature_norm = torch.norm(patch_feats, dim=2)

    # 2D 형태로 변환 및 업샘플링
    h = w = int(np.sqrt(N))
    feature_norm = feature_norm.reshape(-1, 1, h, w)  # Add channel dimension for interpolation

    # 업샘플링
    target_size = (h * patch_size, w * patch_size)
    feature_norm = torch.nn.functional.interpolate(
        feature_norm, size=target_size, mode=mode
    )  # (B, 1, H*patch_size, W*patch_size)

    # 정규화
    return min_max_scale_2d(feature_norm).squeeze(1)

# Model Load

In [ ]:
# 입력이미지
B, C, H, W = data.shape
input_img = data.cpu().numpy().transpose(0, 2, 3, 1)  # (B, H, W, C)
input_img = input_img.reshape(B * H, W, C)
margin = np.ones((input_img.shape[0], 5, 3))

# 모델 로드
freqmae = get_freqmae(
    "your_path/save/fundus_ssl/logs/ssl-freqmae-vit_base_224_14-42/runs/m_0.15_hf_0.1/checkpoints/encoder_decoder-epoch_0000099.ckpt"
)
mae = get_freqmae(
    "your_path/save/fundus_ssl/logs/ssl-freqmae3-vit_base_224_14-42/runs/mae_0.25/checkpoints/encoder_decoder-epoch_0000099.ckpt"
)
# flair = get_flair()
# eva02 = get_eva02()
# dinov2 = get_dinov2()
urfound = get_urfound()
# keepfit = get_keepfit()
retfound = get_retfound()
retclip = get_retclip()

## PCA feats

In [ ]:
# Times New Roman 폰트 설정
plt.rcParams["font.family"] = "Times New Roman"
layer_name = "late"
mode = "bicubic"

# Process data in batches of 4
batch_size = 4
num_batches = (data.shape[0] + batch_size - 1) // batch_size  # Ceiling division
th = 0.0

for batch_idx in range(num_batches):
    # Get current batch
    start_idx = batch_idx * batch_size
    end_idx = min(start_idx + batch_size, data.shape[0])
    batch_data = data[start_idx:end_idx]

    # Get features for current batch
    feats_freqmae = freqmae.get_intermediate_features(batch_data)[layer_name]
    feats_mae = mae.get_intermediate_features(batch_data)[layer_name]
    # feats_flair = flair.get_intermediate_features(batch_data)[layer_name]
    # feats_eva02 = eva02.get_intermediate_features(batch_data)[layer_name]
    # feats_dinov2 = dinov2.get_intermediate_features(batch_data)[layer_name]
    feats_urfound = urfound.get_intermediate_features(batch_data)[layer_name]
    # feats_keepfit = keepfit.get_intermediate_features(batch_data)[layer_name]
    feats_retfound = retfound.get_intermediate_features(batch_data)[layer_name]
    feats_retclip = retclip.get_intermediate_features(batch_data)[layer_name]

    # Apply PCA to features
    pca_feats_freqmae = patch_pca_feats(feats_freqmae[:, 5:], 16, batch_data)
    pca_feats_mae = patch_pca_feats(feats_mae[:, 5:], 16, batch_data)
    # pca_feats_flair = patch_pca_feats(feats_flair, 16, batch_data)
    pca_feats_urfound = patch_pca_feats(feats_urfound[:, 1:], 16, batch_data)
    # pca_feats_keepfit = patch_pca_feats(feats_keepfit[:, 1:], 16, batch_data)
    pca_feats_retfound = patch_pca_feats(feats_retfound[:, 1:], 16, batch_data)
    pca_feats_retclip = patch_pca_feats(feats_retclip[:, 1:], 16, batch_data)

    feats = {
        # "EVA02": pca_feats_eva02,
        # "DINOv2": pca_feats_dinov2,
        # "FLAIR": pca_feats_flair,
        # "KeepFIT": pca_feats_keepfit,
        "UrFound": pca_feats_urfound,
        "RetClip": pca_feats_retclip,
        "RetFound": pca_feats_retfound,
        "MAE": pca_feats_mae,
        "FreqMAE": pca_feats_freqmae,
    }

    # Process input images for current batch
    batch_input_img = batch_data.cpu().numpy().transpose(0, 2, 3, 1)  # (B, H, W, C)
    batch_input_img = batch_input_img.reshape((end_idx - start_idx) * H, W, C)
    margin = np.ones((batch_input_img.shape[0], 5, 3))

    new_feats = {}
    for model, feat in feats.items():
        reshpae_feats = resize(feat, (224, 224))
        reshpae_feats = einops.rearrange(reshpae_feats, "b c h w -> (b h) w c").cpu().numpy()
        new_feats[model] = reshpae_feats

    # Check if input_img is already a tensor
    if isinstance(batch_input_img, torch.Tensor):
        input_img_np = batch_input_img.cpu().numpy()
    else:
        input_img_np = batch_input_img

    # Check if margin is already a tensor
    if isinstance(margin, torch.Tensor):
        margin_np = margin.cpu().numpy()
    else:
        margin_np = margin

    concat_img = [input_img_np, margin_np]
    for feat in new_feats.values():
        concat_img += [feat, margin_np]

    # Concatenate images and visualize
    concat_img = np.concatenate(concat_img, axis=1).clip(0, 1)
    plt.figure(figsize=(15, 10))
    plt.imshow(concat_img)
    plt.title(
        f"Batch {batch_idx+1}/{num_batches}: PCA Features Visualization of {layer_name} layer",
        fontsize=16,
        pad=40,
    )

    # Add model names as subtitles
    plt.text(input_img_np.shape[1] // 2, -20, "Input", ha="center", fontsize=14)

    # Calculate position for each model name
    current_pos = input_img_np.shape[1] + margin_np.shape[1]
    for model_name in new_feats.keys():
        # Position text in the middle of each feature visualization
        text_pos = current_pos + new_feats[model_name].shape[1] // 2
        plt.text(text_pos, -20, model_name, ha="center", fontsize=12)
        # Move to next position (feature width + margin width)
        current_pos += new_feats[model_name].shape[1] + margin_np.shape[1]

    plt.axis("off")
    plt.tight_layout(pad=2.0)
    plt.show()

## Norm Feats

In [ ]:
# Process data in batches of 4
batch_size = 4
num_batches = (data.shape[0] + batch_size - 1) // batch_size  # Ceiling division
layer_name = "late"

for batch_idx in range(num_batches):
    # Get current batch
    start_idx = batch_idx * batch_size
    end_idx = min(start_idx + batch_size, data.shape[0])
    batch_data = data[start_idx:end_idx]

    # Get features for current batch
    feats_freqmae = freqmae.get_intermediate_features(batch_data)[layer_name]
    feats_mae = mae.get_intermediate_features(batch_data)[layer_name]
    # feats_flair = flair.get_intermediate_features(batch_data)[layer_name]
    feats_urfound = urfound.get_intermediate_features(batch_data)[layer_name]
    # feats_keepfit = keepfit.get_intermediate_features(batch_data)[layer_name]
    feats_retfound = retfound.get_intermediate_features(batch_data)[layer_name]
    feats_retclip = retclip.get_intermediate_features(batch_data)[layer_name]

    # Extract patch features (excluding prefix tokens)
    patch_feats_freqmae = feats_freqmae[:, 5:]
    patch_feats_mae = feats_mae[:, 5:]
    # patch_feats_flair = feats_flair
    patch_feats_urfound = feats_urfound[:, 1:]
    # patch_feats_keepfit = feats_keepfit[:, 1:]
    patch_feats_retfound = feats_retfound[:, 1:]
    patch_feats_retclip = feats_retclip[:, 1:]

    # Process all norm features using the patch_norm_feats function
    patch_size = 16  # Assuming patch size is 16

    norm_feats = {
        # "FLAIR": patch_norm_feats(patch_feats_flair, patch_size, mode),
        # "KeepFIT": patch_norm_feats(patch_feats_keepfit, patch_size, mode),
        "UrFound": patch_norm_feats(patch_feats_urfound, patch_size, mode),
        "RetClip": patch_norm_feats(patch_feats_retclip, patch_size, mode),
        "RetFound": patch_norm_feats(patch_feats_retfound, patch_size, mode),
        "MAE": patch_norm_feats(patch_feats_mae, patch_size, mode),
        "FreqMAE": patch_norm_feats(patch_feats_freqmae, patch_size, mode),
    }

    # Process input images for current batch
    batch_input_img = batch_data.cpu().numpy().transpose(0, 2, 3, 1)  # (B, H, W, C)
    batch_input_img = batch_input_img.reshape((end_idx - start_idx) * H, W, C)
    margin = np.ones((batch_input_img.shape[0], 5, 3))

    # Convert norm features to heatmaps
    new_feats = {}
    for model, feat in norm_feats.items():
        heatmaps = []
        for f in feat:
            resized_f = resize(f, (224, 224))
            heatmap = convert_attn_to_heatmap(resized_f, "magma")
            heatmaps.append(heatmap.unsqueeze(0))

        heatmap = torch.cat(heatmaps, dim=0)
        heatmap = einops.rearrange(heatmap, "b c h w -> (b h) w c").cpu().numpy()
        new_feats[model] = heatmap

    # Check if input_img is already a tensor
    if isinstance(batch_input_img, torch.Tensor):
        input_img_np = batch_input_img.cpu().numpy()
    else:
        input_img_np = batch_input_img

    # Check if margin is already a tensor
    if isinstance(margin, torch.Tensor):
        margin_np = margin.cpu().numpy()
    else:
        margin_np = margin

    concat_img = [input_img_np, margin_np]
    for feat in new_feats.values():
        concat_img += [feat, margin_np]

    # Concatenate images and visualize
    concat_img = np.concatenate(concat_img, axis=1).clip(0, 1)
    plt.figure(figsize=(15, 10))
    plt.imshow(concat_img)
    plt.title(
        f"Batch {batch_idx+1}/{num_batches}: Feature Norm Visualization of {layer_name} layer",
        fontsize=16,
        pad=40,
    )

    # Add model names as subtitles
    plt.text(input_img_np.shape[1] // 2, -20, "Input", ha="center", fontsize=14)

    # Calculate position for each model name
    current_pos = input_img_np.shape[1] + margin_np.shape[1]
    for model_name in new_feats.keys():
        # Position text in the middle of each feature visualization
        text_pos = current_pos + new_feats[model_name].shape[1] // 2
        plt.text(text_pos, -20, model_name, ha="center", fontsize=12)
        # Move to next position (feature width + margin width)
        current_pos += new_feats[model_name].shape[1] + margin_np.shape[1]

    plt.axis("off")
    plt.tight_layout(pad=2.0)
    plt.show()